# HPO SolarSystemLander

Study Series 4 (8D) from `hpo/design5.md`.

## Setup

In [ ]:
# Colab setup
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys
from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.drive_backup import backup_to_drive, restore_from_drive
from hpo.lunar_lander.logging import configure_file_logging

drive.mount("/content/drive")
DRIVE_STUDY_DIR = Path("/content/drive/MyDrive/rl_lab/hpo")
LOCAL_STUDY_DIR = Path("/content/rl_lab/hpo/runs")
DRIVE_STUDY_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_STUDY_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
import torch

from dqn.vector_training import VectorTrainingConfig
from hpo.evaluation.dashboard import Dashboard
from hpo.objective import EvaluationConfig, TrialConfig
from hpo.solar_system_lander.environment import EnvFactory
from hpo.study import Baseline, StudyRunner, neighbors

OBSERVATION_MODE = "8d"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRIAL_CFG = TrialConfig(num_envs=20, device=device)
EVALUATION_CFG = EvaluationConfig(eval_episodes=10)
ENVIRONMENT_FACTORY = EnvFactory(OBSERVATION_MODE)
RUN_NAME = "solar_system_lander_8d_series4"
STORAGE_PATH = LOCAL_STUDY_DIR / f"{RUN_NAME}.db"
DRIVE_STORAGE_PATH = DRIVE_STUDY_DIR / f"{RUN_NAME}.db"
LOG_PATH = LOCAL_STUDY_DIR / f"{RUN_NAME}.log"
DRIVE_LOG_PATH = DRIVE_STUDY_DIR / f"{RUN_NAME}.log"
restore_from_drive(DRIVE_STORAGE_PATH, STORAGE_PATH)
restore_from_drive(DRIVE_LOG_PATH, LOG_PATH)
configure_file_logging(LOCAL_STUDY_DIR, LOG_PATH.name)

def backup_study_to_drive():
    backup_to_drive(
        local_database=STORAGE_PATH,
        drive_database=DRIVE_STORAGE_PATH,
        local_log=LOG_PATH,
        drive_log=DRIVE_LOG_PATH,
    )

device, STORAGE_PATH, DRIVE_STORAGE_PATH


## Search spaces

In [ ]:
BATCH_SIZES = [512, 1_024]
LEARNING_STARTS = [2_500, 5_000, 10_000]
OPTIMIZE_EVERY = [4, 8]
EPISODE_COUNTS = [1_000, 1_500, 2_000]
LONG_TRAINING_REPLAY_CAPACITY = 1_200_000

BASELINE_PARAMS = {
    "learning_rate": 0.0022727854024196057,
    "batch_size": 512,
    "eps_end": 0.047716002108220544,
    "eps_decay": 43_214,
    "gamma": 0.99,
    "tau": 0.005,
    "learning_starts": 2_500,
    "optimize_every": 2,
    "replay_memory_capacity": 1_200_000,
    "num_episodes": 500,
}

def vector_config(params):
    return VectorTrainingConfig(
        num_episodes=params["num_episodes"],
        batch_size=params["batch_size"],
        gamma=params["gamma"],
        eps_start=1.0,
        eps_end=params["eps_end"],
        eps_decay=params["eps_decay"],
        tau=params["tau"],
        learning_rate=params["learning_rate"],
        learning_starts=params["learning_starts"],
        optimize_every=params["optimize_every"],
    )

class SearchSpace1:
    def training_config(self, trial, params):
        return vector_config(params | {
            "num_episodes": trial.suggest_categorical(
                "num_episodes", EPISODE_COUNTS
            ),
        })

    def replay_memory_capacity(self, trial, params):
        return LONG_TRAINING_REPLAY_CAPACITY

class SearchSpace2:
    def training_config(self, trial, params):
        return vector_config(params | {
            "eps_end": trial.suggest_float(
                "eps_end",
                0.03,
                0.08,
            ),
            "eps_decay": trial.suggest_int(
                "eps_decay",
                75_000,
                300_000,
                log=True,
            ),
        })

    def replay_memory_capacity(self, trial, params):
        return LONG_TRAINING_REPLAY_CAPACITY

class SearchSpace3:
    def training_config(self, trial, params):
        return vector_config(params | {
            "learning_rate": trial.suggest_float(
                "learning_rate",
                params["learning_rate"] * 0.5,
                params["learning_rate"] * 1.25,
                log=True,
            ),
            "batch_size": trial.suggest_categorical(
                "batch_size", BATCH_SIZES
            ),
            "learning_starts": trial.suggest_categorical(
                "learning_starts", LEARNING_STARTS
            ),
            "optimize_every": trial.suggest_categorical(
                "optimize_every", OPTIMIZE_EVERY
            ),
        })

    def replay_memory_capacity(self, trial, params):
        return LONG_TRAINING_REPLAY_CAPACITY

class SearchSpace4:
    def training_config(self, trial, params):
        return vector_config(params | {
            "learning_rate": trial.suggest_float(
                "learning_rate",
                params["learning_rate"] * 0.75,
                params["learning_rate"] * 1.25,
                log=True,
            ),
            "batch_size": trial.suggest_categorical(
                "batch_size", neighbors(params["batch_size"], BATCH_SIZES)
            ),
            "eps_end": trial.suggest_float(
                "eps_end",
                max(0.02, params["eps_end"] - 0.01),
                min(0.10, params["eps_end"] + 0.01),
            ),
            "eps_decay": trial.suggest_int(
                "eps_decay",
                int(params["eps_decay"] * 0.75),
                int(params["eps_decay"] * 1.25),
                log=True,
            ),
            "learning_starts": trial.suggest_categorical(
                "learning_starts",
                neighbors(params["learning_starts"], LEARNING_STARTS),
            ),
            "optimize_every": trial.suggest_categorical(
                "optimize_every",
                neighbors(params["optimize_every"], OPTIMIZE_EVERY),
            ),
            "num_episodes": trial.suggest_categorical(
                "num_episodes",
                neighbors(params["num_episodes"], EPISODE_COUNTS),
            ),
        })

    def replay_memory_capacity(self, trial, params):
        return LONG_TRAINING_REPLAY_CAPACITY


## Optimization

In [ ]:
runner = StudyRunner(
    database_path=lambda _name: STORAGE_PATH,
    environment_factory=ENVIRONMENT_FACTORY,
    trial_cfg=TRIAL_CFG,
    evaluation_cfg=EVALUATION_CFG,
    baseline=Baseline(params=BASELINE_PARAMS, score=0.0),
    reporter=Dashboard(),
    study_attrs=ENVIRONMENT_FACTORY.metadata(),
    robust_candidates=5,
    extra_seeds=(1001, 1002, 1003, 1004),
    sync_fn=backup_study_to_drive,
)

runner.run("s1_flight_hours", SearchSpace1(), 25)
runner.run("s2_exploration_schedule", SearchSpace2(), 25)
runner.run("s3_learning_regime", SearchSpace3(), 25)
runner.run("s4_joint_finetune", SearchSpace4(), 40)


## Reload studies after a runtime reset

In [ ]:
# Run only after the runtime environment has been reset.
import optuna

def load_study(name):
    return optuna.load_study(study_name=name, storage=f"sqlite:///{STORAGE_PATH}")

study1 = load_study("s1_flight_hours")
study2 = load_study("s2_exploration_schedule")
study3 = load_study("s3_learning_regime")
study4 = load_study("s4_joint_finetune")

## Analysis

In [ ]:
import pandas as pd

studies = [study1, study2, study3, study4]
labels = ["S1", "S2", "S3", "S4"]

rows = []
for label, study in zip(labels, studies):
    rows.append({
        "study": label,
        "score": study.user_attrs["incumbent_score"],
    })

display(pd.DataFrame(rows).set_index("study").style.format("{:.1f}"))